In [1]:
import pyarrow.dataset as ds
import pyarrow.fs as fs
import pyarrow.parquet as pq

In [2]:
filesystem = fs.HadoopFileSystem("hdfs://arnsdpsbx", port=0)

In [3]:
target_path = "/user/team/team_sbertype_evolution/beka_data/dlt/custom_heads/home/targets_test"

In [4]:
remote_path = "/user/team/team_sbertype_evolution/beka_data/dlt/custom_heads/home/dataset_test"

In [5]:
dataset = ds.dataset(remote_path, filesystem=filesystem, format="parquet")

In [6]:
targets = ds.dataset(target_path, filesystem=filesystem, format="parquet")

In [7]:
import polars as pl

In [8]:
pl_targets = pl.from_arrow(targets.to_table())

In [9]:
from tqdm.autonotebook import tqdm

In [10]:
names = ["dir_token", "ecom_token", "mcc_token", "brand_token", "city_token", "amt_token"]
all_names = ["date_stamp", *names]

In [11]:
def handle_batch(pl_batch: pl.DataFrame, target_col: str = "vacation_home_d30") -> pl.DataFrame:
    result: pl.DataFrame = (
        pl_batch.with_columns(pl.col("report_dt").dt.timestamp("ms").floordiv(1000))
        .with_columns(
            pl.lit([])
            .list.concat(
                [
                    pl.col("value").fill_null([]),
                    pl.col("vnv_value").fill_null([]),
                    pl.col("okko_value").fill_null([]),
                    pl.col("samokat_value").fill_null([]),
                ]
            )
            .alias("all_values"),
        )
        .select("epk_id", "report_dt", "all_values", "vacation_home_d30")
        .with_columns(
            pl.col("all_values").list.eval(pl.element().list.get(j, null_on_oob=True).fill_null(0)).alias(name)
            for j, name in enumerate(all_names)
        )
        .select("epk_id", "report_dt", target_col, *all_names)
        .explode(all_names)
        .sort("date_stamp", "epk_id", "report_dt", maintain_order=True)
        .group_by(
            ["epk_id", "report_dt", "vacation_home_d30"],
            maintain_order=True,
        )
        .agg(*all_names)
        .with_columns(pl.col("date_stamp").list.concat(pl.col("report_dt")))
        .with_columns((pl.col(name).list.concat(pl.lit(3)) for j, name in enumerate(names)))
        .select("epk_id", "report_dt", "vacation_home_d30", *all_names)
    )

    return result

In [12]:
!rm -r /home/datalab/nfs/romashka_test_data
!mkdir -p /home/datalab/nfs/romashka_test_data

In [13]:
for i, batch in enumerate(tqdm(dataset.to_batches())):
    pl_batch = pl.from_arrow(batch).join(pl_targets, on=("epk_id", "report_dt"))

    pl_result = handle_batch(pl_batch)

    result_name: str = f"/home/datalab/nfs/romashka_test_data/batch_{i:04d}.parquet"
    pq.write_table(pl_result.to_arrow(), result_name)

0it [00:00, ?it/s]